# 00C — SQL + Python Workbench
Consola analítica libre para trabajar con PostgreSQL real desde VS Code.


In [ ]:
from __future__ import annotations
import sys
from pathlib import Path
import pandas as pd
import numpy as np

cwd=Path.cwd().resolve()
PROJECT_ROOT=cwd if (cwd/"pyproject.toml").exists() else cwd.parent
if not (PROJECT_ROOT/"pyproject.toml").exists():
    raise RuntimeError("Ejecuta este notebook dentro de bd_replica_crm.")
SRC=PROJECT_ROOT/"src"
if str(SRC) not in sys.path:
    sys.path.insert(0,str(SRC))
from replica_cygnus.settings import load_settings
from replica_cygnus.connections import connect_postgres
settings=load_settings(PROJECT_ROOT)
conn=connect_postgres(settings)
pd.set_option("display.max_columns",160)
pd.set_option("display.max_rows",160)
pd.set_option("display.width",240)
def sql_df(sql,params=None):
    return pd.read_sql_query(sql,conn,params=params)
print("DB:",settings.postgres.database)


## 1. SQL libre


In [ ]:
QUERY="""SELECT table_schema,table_name FROM information_schema.tables WHERE table_schema NOT IN (\'pg_catalog\',\'information_schema\') ORDER BY 1,2"""
result=sql_df(QUERY)
result


## 2. Preview de cualquier tabla


In [ ]:
def preview(schema,table,limit=20):\n    return sql_df(f\'SELECT * FROM "{schema}"."{table}" LIMIT {int(limit)}\')


## 3. Perfil rápido


In [ ]:
def profile(frame):\n    rows=[]\n    for c in frame.columns:\n        s=frame[c]\n        rows.append({"column":c,"dtype":str(s.dtype),"null_pct":s.isna().mean(),"nunique":s.nunique(dropna=True)})\n    return pd.DataFrame(rows).sort_values("null_pct",ascending=False)


## 4. SQL → pandas


In [ ]:
lead_sample=sql_df("""SELECT * FROM features.lead_evidence ORDER BY decision_at DESC LIMIT 5000""")\ndisplay(lead_sample.head())\ndisplay(profile(lead_sample).head(30))


## 5. Zona libre de modelado


In [ ]:
# from statsmodels.formula.api import ols, logit\n# from sklearn.linear_model import LogisticRegression


In [ ]:
conn.close(); print("Conexión cerrada.")
